# Phase 4 — A1 / Full fine-tuning (independent Kaggle notebook)

**What this notebook does:** Update the decoder backbone and its new two-class head.

**One notebook = one method = one dedicated Kaggle output folder and one private HF repository.** There is no METHOD switch to change after a run. Save a Kaggle version to retain its outputs.

**Code flow:** install → settings → account login → architecture → source checkpoint → dataset and duplicate-ID audit → chunks → classifier → training → evaluation → test predictions → submission ZIP → private Hugging Face upload.

Input: official `Phase_4_CJPE` train/dev/test. Output: `prediction.json` with one binary integer per official test ID.

**Do not publish until the real Phase-1 checkpoints and official data have been checked in Kaggle.**

## 1 · Install dependencies

Packages needed for the official dataset, tokenizer, checkpoints, evaluation, and optional Hugging Face publishing.

In [1]:
%pip -q install "transformers>=4.45,<5" "huggingface_hub>=0.31,<2" "safetensors>=0.4.5" scikit-learn tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## 2 · Locked experiment and configuration

**Method for this file: `full` (Full fine-tuning).** Keep it fixed to preserve an independent experiment. You can adjust resource-related settings, but changing the method should mean opening its *other* notebook. The run generates its own output directory and dedicated Hugging Face repository.

In [2]:
# One fixed training method in this independent notebook.
ARCH = 'a1'
METHOD = 'full'    # FIXED: open the other notebook for other training methods

# Source checkpoint locations were transcribed from your earlier notebooks.
# Logged-in Hugging Face username is found automatically below.
SOURCE_REPO_NAME = 'cs781-a1-checkpoints'
CHECKPOINT_FILE = 'checkpoint-6104/model.safetensors'
# Optional manual override when the original repo lives under a DIFFERENT owner:
SOURCE_REPO_OVERRIDE = 'P-261110044/cs781-a1-checkpoints'  # Source owner from YOUR Phase-2 notebook
TEACHER_REPO_NAME = 'cs781-phase4-a3-full'  # Student only: run A3 teacher notebook first

CHUNK_LENGTH = 192          # <= base model's positional/context limit
MAX_CHUNKS = 3             # evenly spaced: covers start, middle and end
EPOCHS, PATIENCE = 3, 2  # Stop early if dev Macro-F1 does not improve
GRAD_ACCUM = 4
LR_BACKBONE, LR_HEAD = 2e-5, 2e-4
LR_ADAPT = 1e-4
LORMA_RANK, LORMA_ALPHA = 8, 16
ADAPTER_WIDTH = 48
STUDENT_LAYERS = 4          # only used for A3/KD
KD_T = 2.0
SEED = 42

# None means the complete official split. NEVER publish capped debug predictions.
MAX_TRAIN = MAX_DEV = MAX_TEST = None
UPLOAD_TO_HF = True        # uploads only after predictions & checks succeed
LOCAL_DATA_ROOT = ''       # optional Kaggle Input directory containing Phase_4_CJPE/
LOCAL_MODEL_ROOT = ''      # optional copy of your Phase-1 HF repo as directory
# Macro-F1 gives equal importance to both binary classes.
# Weighted cross-entropy is optional; weights are computed ONLY from training labels.
USE_CLASS_WEIGHTS = True

## 3 · Imports, GPU, random seed, output folder

Imports and local output management are isolated so you can distinguish setup errors from training errors.

In [3]:
# Reproducibility, hardware selection, and run-specific output directory.
import os, json, re, math, time, zipfile, random, shutil, gc
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from transformers import PreTrainedTokenizerFast
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import load_file
from tqdm.auto import tqdm

assert METHOD in ('full', 'adapters'), f'Unsupported method for architecture {ARCH}: {METHOD}'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = Path('/kaggle/working' if Path('/kaggle/working').is_dir() else '.') / f'phase4_{ARCH}_{METHOD}'
OUT.mkdir(exist_ok=True,parents=True)

# Kaggle secret name may be HF_TOKEN or HF; neither the token nor its value is printed.


## 4 · Authenticate Hugging Face and identify your account automatically

Reads the token from the **Kaggle Secret** named `HF_TOKEN` (or `HF`). Your account name is detected by the API. The **input checkpoint repository** is deliberately kept separate from your output account.

In [4]:
# The token is never printed; input repo comes from your Phase-1 history.
TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for key in ('HF_TOKEN','HF'):
            try:
                TOKEN = secrets.get_secret(key)
                if TOKEN: break
            except Exception: pass
    except ImportError: pass
if not TOKEN:
    raise RuntimeError('Set Kaggle Secret HF_TOKEN (with HF read/write permissions). Internet must be on.')
api = HfApi(token=TOKEN)
HF_USER = api.whoami()['name']        # automatic identity, never hardcoded
P1_REPO = SOURCE_REPO_OVERRIDE or f'{HF_USER}/{SOURCE_REPO_NAME}'
OUT_REPO = f'{HF_USER}/cs781-phase4-{ARCH}-{METHOD}'
print('Source:', P1_REPO, 'Output repo:', OUT_REPO, 'Device:', DEVICE)


Source: P-261110044/cs781-a1-checkpoints Output repo: P-261110044/cs781-phase4-a1-full Device: cuda


## 5 · Original Phase 1 architecture — copied intact

These classes define the exact Phase 1 backbone; their parameter names/shapes must match the saved checkpoint. **Do not replace them with a pretrained public model.** The new classification head is defined later.

In [5]:
# Copied from a1_full.ipynb; no pretrained external model substituted.
from transformers.modeling_outputs import CausalLMOutput
class DecoderBlock(nn.Module):
    """
    One Transformer decoder block.

    Structure:
        LayerNorm
        -> Causal Multi-Head Self Attention
        -> Residual Connection
        -> LayerNorm
        -> Feed Forward Network
        -> Residual Connection
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        # Normalize before self-attention.
        self.norm1 = nn.LayerNorm(
            d_model
        )

        # Multi-head self-attention.
        #
        # Query, Key and Value all come from
        # the same token sequence.
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        # Normalize before feed-forward network.
        self.norm2 = nn.LayerNorm(
            d_model
        )

        # Transformer feed-forward network.
        self.feed_forward = nn.Sequential(
            nn.Linear(
                d_model,
                d_ff
            ),
            nn.GELU(),
            nn.Linear(
                d_ff,
                d_model
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        causal_mask
    ):

        # ----------------------------------------------------
        # 1. Causal self-attention
        # ----------------------------------------------------

        z = self.norm1(x)

        attention_output, _ = self.attention(
            z,
            z,
            z,
            attn_mask=causal_mask,
            need_weights=False
        )

        # Residual connection.
        x = x + self.dropout(
            attention_output
        )

        # ----------------------------------------------------
        # 2. Feed-forward network
        # ----------------------------------------------------

        z = self.norm2(x)

        ff_output = self.feed_forward(
            z
        )

        # Second residual connection.
        x = x + self.dropout(
            ff_output
        )

        return x

class DecoderTransformerLM(nn.Module):
    """
    Complete custom decoder-only Transformer
    language model.
    """

    def __init__(
        self,
        vocab_size,
        context_length=2048,
        d_model=640,
        n_heads=10,
        n_layers=8,
        d_ff=2560,
        dropout=0.1
    ):
        super().__init__()

        self.context_length = (
            context_length
        )

        self.vocab_size = (
            vocab_size
        )

        # Safety check:
        # embedding dimension must divide evenly
        # across attention heads.
        assert (
            d_model % n_heads == 0
        ), (
            "d_model must be divisible "
            "by n_heads"
        )

        # ----------------------------------------------------
        # Token embeddings
        # ----------------------------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # ----------------------------------------------------
        # Learned positional embeddings
        # ----------------------------------------------------

        self.position_embedding = nn.Embedding(
            context_length,
            d_model
        )

        # ----------------------------------------------------
        # Stack of custom Transformer decoder blocks
        # ----------------------------------------------------

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization.
        self.final_norm = nn.LayerNorm(
            d_model
        )

        # Convert hidden representation
        # into vocabulary logits.
        self.output_layer = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )


    def forward(
        self,
        input_ids,
        labels=None,
        attention_mask=None,
        **kwargs
    ):

        _, seq_len = input_ids.shape

        if seq_len > self.context_length:
            raise ValueError(
                f"Sequence length {seq_len} exceeds "
                f"context length {self.context_length}"
            )

        # ----------------------------------------------------
        # 1. Token embeddings
        # ----------------------------------------------------

        token_vectors = (
            self.token_embedding(
                input_ids
            )
        )

        # ----------------------------------------------------
        # 2. Positional embeddings
        # ----------------------------------------------------

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        position_vectors = (
            self.position_embedding(
                positions
            )
        )

        x = (
            token_vectors
            +
            position_vectors
        )

        # ----------------------------------------------------
        # 3. Causal mask
        #
        # True = attention blocked.
        #
        # Token i cannot see tokens after i.
        # ----------------------------------------------------

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # ----------------------------------------------------
        # 4. Transformer blocks
        # ----------------------------------------------------

        for block in self.blocks:

            x = block(
                x,
                causal_mask
            )

        # ----------------------------------------------------
        # 5. Vocabulary prediction
        # ----------------------------------------------------

        x = self.final_norm(
            x
        )

        logits = self.output_layer(
            x
        )

        loss = None

        # ----------------------------------------------------
        # 6. Next-token language modelling loss
        # ----------------------------------------------------

        if labels is not None:

            # Example:
            #
            # logits:
            # t0 t1 t2
            #
            # targets:
            # t1 t2 t3

            shift_logits = (
                logits[:, :-1, :]
                .contiguous()
            )

            shift_labels = (
                labels[:, 1:]
                .contiguous()
            )

            loss = F.cross_entropy(
                shift_logits.view(
                    -1,
                    shift_logits.size(-1)
                ),
                shift_labels.view(-1)
            )

        return CausalLMOutput(
            loss=loss,
            logits=logits
        )


## 6 · Locate the original checkpoint and tokenizer

Fetch weights and BPE tokenizer together, reject missing files, and ensure the tokenizer vocabulary equals the embedding matrix rows. The notebook never silently adds tokens.

In [6]:
# Find source files, copy the original tokenizer, and read the exact saved weights.
# ---- Official Phase-1 checkpoint; exact architecture and vocabulary ----
repo_files = ([str(f.relative_to(LOCAL_MODEL_ROOT)).replace('\\','/')
               for f in Path(LOCAL_MODEL_ROOT).rglob('*') if f.is_file()]
              if LOCAL_MODEL_ROOT else api.list_repo_files(P1_REPO, repo_type='model'))
if CHECKPOINT_FILE not in repo_files:
    raise FileNotFoundError(f'Expected {CHECKPOINT_FILE} in {P1_REPO}, available: {repo_files[:40]}')

def fetch(rel):
    return (Path(LOCAL_MODEL_ROOT)/rel if LOCAL_MODEL_ROOT else
            Path(hf_hub_download(repo_id=P1_REPO,filename=rel,token=TOKEN)))

parent = str(Path(CHECKPOINT_FILE).parent)
parent = '' if parent == '.' else parent+'/'
folders = list(dict.fromkeys([parent,'','final_model/','architecture-3-final/',
                              'architecture-4-final/', 'cs781_phase1_a2_encoderdecoder/legal-gpt2-final/']))
tokenizer_folder = next((f for f in folders if f+'tokenizer.json' in repo_files),None)
if tokenizer_folder is None: raise FileNotFoundError('Original Phase-1 tokenizer.json not found')
tok_dir = OUT/'tokenizer'; tok_dir.mkdir(exist_ok=True)
for name in ('tokenizer.json','tokenizer_config.json','special_tokens_map.json','added_tokens.json'):
    if tokenizer_folder+name in repo_files:
        shutil.copy2(fetch(tokenizer_folder+name),tok_dir/name)
tokenizer = PreTrainedTokenizerFast.from_pretrained(str(tok_dir))
# Never add a token silently: changing tokenizer length breaks checkpoint compatibility.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
if tokenizer.pad_token_id is None: raise ValueError('Original tokenizer has no EOS/UNK/PAD token')

config_name = next((n for n in (parent+'architecture_config.json','architecture_config.json') if n in repo_files),None)
saved_cfg = json.loads(fetch(config_name).read_text()) if config_name else {}
pth = fetch(CHECKPOINT_FILE)
state = load_file(str(pth),device='cpu') if pth.suffix=='.safetensors' else torch.load(pth,map_location='cpu',weights_only=True)
for key in ('model_state','state_dict','model_state_dict'):
    if key in state and isinstance(state[key],dict): state=state[key]; break
for prefix in ('module.','_orig_mod.'):
    if state and all(k.startswith(prefix) for k in state):
        state={k[len(prefix):]:v for k,v in state.items()}
if 'token_embedding.weight' not in state:
    raise ValueError('Checkpoint does not have the Phase-1 token_embedding.weight key')
VOCAB,DIM = state['token_embedding.weight'].shape
if len(tokenizer) != VOCAB: raise ValueError(f'Tokenizer mismatch {len(tokenizer)} vs checkpoint {VOCAB}')


tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'TokenizersBackend'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


checkpoint-6104/model.safetensors:   0%|          | 0.00/419M [00:00<?, ?B/s]

## 7 · Construct and strictly load the Phase 1 backbone

Infer model dimensions from weights, check head/pattern settings, and reject missing or mismatched parameters. Remove the vocabulary-output layer **after** loading because Phase 4 needs hidden features rather than next-token probabilities.

In [7]:
# Build the correct Phase-1 class, then check strict state-dict compatibility.
def layer_count(prefix):
    found={int(m.group(1)) for k in state if (m:=re.match(re.escape(prefix)+r'\.(\d+)\.',k))}
    if not found or found != set(range(max(found)+1)): raise ValueError(f'Invalid {prefix} blocks: {found}')
    return len(found)

def heads(arch,dim,n):
    recorded = saved_cfg.get('num_heads',saved_cfg.get('n_heads'))
    if recorded: return int(recorded)
    if arch=='a1' and (dim,n)==(640,8): return 10
    if arch=='a2' and (dim,n)==(600,8): return 15
    raise ValueError('Attention head count missing from config; add num_heads to Phase-1 architecture_config.json')

def make_backbone(student_layers=None):
    n = layer_count('blocks') if student_layers is None else student_layers
    return DecoderTransformerLM(vocab_size=VOCAB,context_length=state['position_embedding.weight'].shape[0],
        d_model=DIM,n_heads=heads(ARCH,DIM,layer_count('blocks')),n_layers=n,
        d_ff=state['blocks.0.feed_forward.0.weight'].shape[0],dropout=0.1)

def strict_load(m, student_layers=None):
    target = m.state_dict()
    sub = {k:v for k,v in state.items() if k in target} if student_layers else dict(state)
    for a,b in (('lm_head.weight','token_embedding.weight'),('token_embedding.weight','lm_head.weight')):
        if a in target and a not in sub and b in sub and target[a].shape == sub[b].shape:
            sub[a] = sub[b]
    if 'lm_head.weight' in sub and 'lm_head.weight' not in target and 'output_layer.weight' in target:
        sub.setdefault('output_layer.weight', sub.pop('lm_head.weight'))
    missing = set(target)-set(sub); extra = set(sub)-set(target)
    invalid = [k for k in set(target)&set(sub) if target[k].shape != sub[k].shape]
    if missing or extra or invalid:
        raise ValueError(f'Strict checkpoint mismatch: missing={sorted(missing)[:5]}, extra={sorted(extra)[:5]}, shape={invalid[:5]}')
    m.load_state_dict(sub,strict=True)
    if ARCH in ('a1','a2'): m.output_layer=nn.Identity()
    else: m.lm_head=nn.Identity()
    return m

backbone = strict_load(make_backbone())
print('Exact Phase-1 backbone loaded:',ARCH,'width:',DIM,'vocab:',VOCAB)


Exact Phase-1 backbone loaded: a1 width: 640 vocab: 50000


## 8 · Load official Phase 4 train, development, and test cases

Try `LOCAL_DATA_ROOT` first if you attached the files as a Kaggle Dataset; otherwise fetch from the official Hugging Face dataset. Check IDs and labels before any training.

**Why the old loader failed:** Phase 4 may contain several TRAIN instances with the same case ID. This cell accepts repeated TRAIN IDs but checks inter-split ID and exact-text overlap as potential leakage. Use the standalone audit notebook for details before relaxing those checks.

In [8]:
# Official Phase 4 files. Repeated TRAIN case IDs can occur when several
# instances/passages of the same case are included in the training split.
# We preserve these records for inspection and training; repeated IDs by
# themselves are NOT proof of train/test leakage.
import hashlib
from collections import defaultdict

DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
paths = None

# Pin all 3 splits to ONE revision, so the files cannot change mid-download.
# When using attached local files, the revision is not needed.
DATA_REVISION = api.dataset_info(DATA_REPO).sha if not LOCAL_DATA_ROOT else None
print('Official dataset revision:', DATA_REVISION or '(local Kaggle dataset)')

def data_file(split):
    name = f'{split}.json'
    folder = 'Test' if split == 'test' else 'Train'
    if LOCAL_DATA_ROOT:
        root = Path(LOCAL_DATA_ROOT)
        options = (root/'Phase_4_CJPE'/folder/name, root/folder/name, root/name)
        for p in options:
            if p.is_file(): return p
        raise FileNotFoundError(f'Official Phase 4 file not found: {options}')

    global paths
    if paths is None:
        paths = api.list_repo_files(DATA_REPO, repo_type='dataset', revision=DATA_REVISION)
    matches = [p for p in paths if ('phase_4' in p.lower()
               and p.lower().endswith('/'+folder.lower()+'/'+name))]
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected one {folder}/{name}; found {matches}')
    return Path(hf_hub_download(repo_id=DATA_REPO,filename=matches[0],
              repo_type='dataset',revision=DATA_REVISION,token=TOKEN))

def load_cases(split):
    """Preserve every training instance; require test IDs to be unique."""
    raw = json.loads(data_file(split).read_text(encoding='utf-8'))
    if isinstance(raw, dict) and isinstance(raw.get('data'), list): raw = raw['data']
    if isinstance(raw, dict):
        raw = [dict(v,id=k) if isinstance(v,dict) else {'id':k,'text':v}
               for k,v in raw.items()]
    if not isinstance(raw, list): raise ValueError(f'{split}: expected a list or ID-keyed dictionary')
    out,seen=[],set()
    for item in raw:
        if 'id' not in item or 'text' not in item:
            raise ValueError(f'{split}: missing id/text; keys={list(item)}')
        ident = str(item['id']).strip()
        if not ident or ident.lower()=='none':
            raise ValueError(f'{split}: empty/invalid case ID')
        if split == 'test' and ident in seen:
            raise ValueError(f'test: ID {ident} repeats; cannot submit one prediction per record')
        seen.add(ident)
        txt=item['text']
        if isinstance(txt,list): txt='\n'.join(txt)
        if not isinstance(txt,str) or not txt.strip():
            raise ValueError(f'{split}: invalid text for {ident}')
        case={'id':ident,'text':txt}
        if split != 'test':
            keys=[k for k in ('label','labels','outcome','judgment','judgement','target','y') if k in item]
            if len(keys)!=1:
                raise ValueError(f'{split} {ident}: expected one binary label key; found {keys}; keys={list(item)}')
            y=item[keys[0]]
            if type(y)!=int or y not in (0,1):
                raise ValueError(f'{split} {ident}: expected integer 0/1; got {y!r}')
            case['label']=y
        out.append(case)
    return out

# First load all data, THEN audit repeated IDs and overlap across splits.
train_cases,dev_cases,test_cases = (load_cases(s) for s in ('train','dev','test'))

# Within-split duplicates are reported but do not stop TRAIN from loading.
for split, cases in [('train',train_cases),('dev',dev_cases),('test',test_cases)]:
    counts=Counter(c['id'] for c in cases)
    duplicates={k:v for k,v in counts.items() if v>1}
    print(f'{split}: {len(cases)} rows, {len(counts)} unique IDs, '
          f'{len(duplicates)} repeated IDs, {sum(v-1 for v in duplicates.values())} extra rows')
    if duplicates:
        print('  Examples:',list(duplicates.items())[:5])
        if split=='dev':
            print('  WARNING: Dev contains multiple records for a case; check metric interpretation.')
    if split!='test':
        labels=defaultdict(set)
        for c in cases: labels[c['id']].add(c['label'])
        conflicting=[k for k,v in labels.items() if len(v)>1]
        if conflicting:
            print(f'  WARNING: {len(conflicting)} case IDs have conflicting labels, e.g. {conflicting[:5]}')

ids={name:set(c['id'] for c in cases) for name,cases in
     [('train',train_cases),('dev',dev_cases),('test',test_cases)]}
for a,b in [('train','dev'),('train','test'),('dev','test')]:
    shared=ids[a]&ids[b]
    print(f'{a}/{b} shared case IDs: {len(shared)}',sorted(shared)[:5])
    if shared:
        raise ValueError(f'POTENTIAL LEAKAGE: {len(shared)} IDs shared by {a} and {b}. '
                         'Inspect the standalone dataset-audit notebook before training.')

# Exact same text under DIFFERENT IDs is also potential leakage.
# Report hashes (rather than comparing hundreds of thousands of text strings).
def text_hash(x):
    normalized=' '.join(x.lower().split())
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()
texts={name:set(text_hash(c['text']) for c in cases) for name,cases in
       [('train',train_cases),('dev',dev_cases),('test',test_cases)]}
for a,b in [('train','dev'),('train','test'),('dev','test')]:
    shared=texts[a]&texts[b]
    print(f'{a}/{b} identical normalized texts: {len(shared)}')
    if shared:
        raise ValueError(f'POTENTIAL LEAKAGE: {len(shared)} matching texts between {a}/{b}. '
                         'Inspect the standalone dataset-audit notebook before training.')

TEST_IDS=[c['id'] for c in test_cases]
print('Training label counts:',Counter(c['label'] for c in train_cases))
# Only cap AFTER auditing all official records. Never submit capped test predictions.
train_cases=train_cases[:MAX_TRAIN] if MAX_TRAIN is not None else train_cases
dev_cases=dev_cases[:MAX_DEV] if MAX_DEV is not None else dev_cases
test_cases=test_cases[:MAX_TEST] if MAX_TEST is not None else test_cases


Official dataset revision: 93b0059be756c6c461f3a225de25aabda7d5f84a


Phase_4_CJPE/Train/train.json:   0%|          | 0.00/737M [00:00<?, ?B/s]

Phase_4_CJPE/Train/dev.json:   0%|          | 0.00/45.0M [00:00<?, ?B/s]

Phase_4_CJPE/Test/test.json:   0%|          | 0.00/35.3M [00:00<?, ?B/s]

train: 37387 rows, 32305 unique IDs, 5082 repeated IDs, 5082 extra rows
  Examples: [('2019_890', 2), ('2014_170', 2), ('2010_721', 2), ('2008_1460', 2), ('2008_188', 2)]
dev: 1988 rows, 994 unique IDs, 994 repeated IDs, 994 extra rows
  Examples: [('1989_75', 2), ('1959_102', 2), ('1979_76', 2), ('1991_124', 2), ('1985_233', 2)]
test: 1517 rows, 1517 unique IDs, 0 repeated IDs, 0 extra rows
train/dev shared case IDs: 0 []
train/test shared case IDs: 0 []
dev/test shared case IDs: 0 []
train/dev identical normalized texts: 0
train/test identical normalized texts: 0
dev/test identical normalized texts: 0
Training label counts: Counter({0: 22067, 1: 15320})


## 9 · Split long legal cases into representative chunks

The transformer context is limited. Instead of truncating every case to its first few words, select evenly spaced chunks from the **start, middle, and end**. A case contributes **one** classification loss, not one loss per chunk. You can compare other chunk selection strategies later as an ablation.

In [9]:
# Tokenize once. Reduce each long document to at most MAX_CHUNKS chunks.
# Pre-tokenize exactly once per case, keep first and last document information.
if ARCH=='a1': model_context=backbone.context_length
elif ARCH=='a2': model_context=backbone.source_position.num_embeddings
elif ARCH=='a4': model_context=backbone.context_length
else: model_context=CHUNK_LENGTH  # A3 implements internal local windows
if CHUNK_LENGTH > model_context: raise ValueError('CHUNK_LENGTH exceeds source model context')

def chunks(text):
    ids=tokenizer.encode(text,add_special_tokens=False)
    if not ids: ids=[tokenizer.unk_token_id]
    windows=[ids[i:i+CHUNK_LENGTH] for i in range(0,len(ids),CHUNK_LENGTH)]
    if len(windows)>MAX_CHUNKS:
        idx=np.linspace(0,len(windows)-1,MAX_CHUNKS,dtype=int).tolist()
        windows=[windows[i] for i in idx]
    return [torch.tensor(x,dtype=torch.long) for x in windows]

# Preserve official case order for submission and dev metrics.
train_data=[(chunks(x['text']),x.get('label'),x['id']) for x in train_cases]
dev_data=[(chunks(x['text']),x.get('label'),x['id']) for x in dev_cases]
test_data=[(chunks(x['text']),None,x['id']) for x in test_cases]


## 10 · Define the task-specific classifier and adaptation layers

The small binary head consumes one pooled representation per case. A1 supports full and adapters; A2 partial and BitFit; A3 full and distillation; A4 LORMA. When a method freezes the base, the head remains trainable.

In [10]:
# Freeze/update only the intended weights and pool chunk representations.
# ---- Classification head + source-inspired adaptation methods ----
class Adapter(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.down=nn.Linear(width,ADAPTER_WIDTH)
        self.up=nn.Linear(ADAPTER_WIDTH,width)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,x): return self.up(F.gelu(self.down(x)))

class LoRMALinear(nn.Module):
    # Matches the Phase-2 A4 method: multiplicative low-rank correction on W(x).
    def __init__(self,original):
        super().__init__()
        self.original=original
        self.original.requires_grad_(False)
        self.A=nn.Linear(original.out_features,LORMA_RANK,bias=False)
        self.B=nn.Linear(LORMA_RANK,original.out_features,bias=False)
        nn.init.zeros_(self.B.weight)
    def forward(self,x):
        z=self.original(x)
        return z+self.B(self.A(z))*(LORMA_ALPHA/LORMA_RANK)

class Classifier(nn.Module):
    def __init__(self,base,method):
        super().__init__()
        self.base=base; self.method=method
        self.head=nn.Linear(DIM,2)
        self._hooks=[]
        if method not in ('full','kd'): self.base.requires_grad_(False)
        blocks=self.base.encoder_blocks if ARCH=='a2' else self.base.blocks
        if method=='partial':
            for layer in blocks[-2:]: layer.requires_grad_(True)
        elif method=='bitfit':
            for n,p in self.base.named_parameters():
                if n.endswith('bias'): p.requires_grad_(True)
        elif method=='adapters':
            self.adapters=nn.ModuleList([Adapter(DIM) for _ in blocks])
            for block,adapter in zip(blocks,self.adapters):
                def add_adapter(mod,args,out,adapter=adapter):
                    if isinstance(out,tuple): return (out[0]+adapter(out[0]),*out[1:])
                    return out+adapter(out)
                self._hooks.append(block.register_forward_hook(add_adapter))
        elif method=='lorma':
            for block in blocks:
                block.attn.q_proj=LoRMALinear(block.attn.q_proj)
                block.attn.v_proj=LoRMALinear(block.attn.v_proj)

    def feature(self,ids):
        # One chunk -> one contextual representation. Only the correct backbone method is called.
        x=ids.to(DEVICE).unsqueeze(0)
        if ARCH=='a1': hidden=self.base(x).logits
        elif ARCH=='a2': hidden=self.base.encode(x)
        else: hidden,_=self.base._run_hidden(x)
        return hidden[:,-1,:].squeeze(0) if ARCH!='a2' else hidden.mean(1).squeeze(0)

    def forward(self,parts):
        # Average chunk vectors so the final decision is at CASE level, not sentence level.
        features=torch.stack([self.feature(x) for x in parts])
        return self.head(features.mean(0))


## 11 · Initialize this method’s classification model

Update the decoder backbone and its new two-class head. No teacher model is needed in this experiment. The `Classifier` class from Step 10 applies the chosen method; this step prints the number of parameters.

In [11]:
# This method does not need another Phase-4 teacher checkpoint.
# The classifier applies exactly the one method selected in Cell 2.
soft_targets = None
model = Classifier(backbone, METHOD).to(DEVICE)
print('Method:', METHOD)
print('Parameters:', sum(p.numel() for p in model.parameters()))
print('Trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad))


Method: full
Parameters: 72701442
Trainable: 72701442


## 12 · Define optimizer and optional training-only class weights

Class imbalance can hurt Macro-F1. Class weights are derived strictly from training labels and supplied to the supervised component of the loss; the developer should compare weighted/unweighted as an ablation. The notebook keeps the head learning rate separate from adapted backbone weights.

In [12]:
# Separate parameter groups + class-aware loss; no dev/test statistics leak into the optimizer.
# ---- Train, keep best dev Macro-F1 checkpoint, then predict ALL official test IDs ----
base_params=[p for n,p in model.named_parameters() if not n.startswith('head.') and p.requires_grad]
groups=[{'params':base_params,'lr':LR_BACKBONE if METHOD in ('full','partial','kd') else LR_ADAPT}] if base_params else []
groups.append({'params':model.head.parameters(),'lr':LR_HEAD})
optimizer=torch.optim.AdamW(groups,weight_decay=.01)

# Compute weights using TRAIN split only — never from dev/test labels.
from sklearn.utils.class_weight import compute_class_weight
train_labels = np.array([y for _, y, _ in train_data], dtype=int)
if USE_CLASS_WEIGHTS and set(train_labels.tolist()) == {0, 1}:
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_labels)
    CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
else:
    CLASS_WEIGHTS = None
print('Training class weights:', CLASS_WEIGHTS)


Training class weights: tensor([0.8471, 1.2202], device='cuda:0')


## 13 · Define held-out evaluation (Macro-F1, precision, recall, confusion matrix)

This cell **only defines** evaluation; it does not train or publish anything. It can be rerun to inspect results. The scoring labels are explicitly `[0, 1]`.

In [13]:
# Predict one binary class per complete case without updating weights.
@torch.no_grad()
def evaluate(data):
    model.eval()
    preds=[int(model(parts).argmax()) for parts,_,_ in tqdm(data,desc='Predict',leave=False)]
    if any(y is None for _,y,_ in data): return None,preds
    ys=[int(y) for _,y,_ in data]
    return {'macro_f1':float(f1_score(ys,preds,labels=[0,1],average='macro',zero_division=0)),
             'precision_macro':float(precision_score(ys,preds,labels=[0,1],average='macro',zero_division=0)),
             'recall_macro':float(recall_score(ys,preds,labels=[0,1],average='macro',zero_division=0)),
             'confusion_matrix':confusion_matrix(ys,preds,labels=[0,1]).tolist()}, preds


## 14 · Train, validate after each epoch, and save the best checkpoint

Gradient accumulation helps small GPUs. Selection uses **development Macro-F1**, not training loss. A3 KD uses both true labels and teacher probabilities; standard methods use the true labels. Output includes a checkpoint and readable epoch history.

In [14]:
# Train on train split only; save weights whenever the dev Macro-F1 improves.
# Reset history when starting a fresh training experiment.
history=[]; best_score=-1; stale=0
best_path=OUT/'best_model.pt'
for ep in range(1,EPOCHS+1):
    model.train(); losses=[]
    order=np.random.RandomState(SEED+ep).permutation(len(train_data))
    optimizer.zero_grad(set_to_none=True)
    for step,j in enumerate(tqdm(order,desc=f'Epoch {ep}'),1):
        j=int(j); parts,y,_=train_data[j]
        logits=model(parts)
        hard=F.cross_entropy(logits.unsqueeze(0),torch.tensor([y],device=DEVICE),weight=CLASS_WEIGHTS)
        if METHOD=='kd':
            teacher_logits=soft_targets[j].to(DEVICE)
            soft=F.kl_div(F.log_softmax(logits/KD_T,dim=-1),
                          F.softmax(teacher_logits/KD_T,dim=-1),reduction='sum')*KD_T**2
            loss=.5*hard+.5*soft
        else: loss=hard
        losses.append(float(loss.detach().cpu()))
        (loss/GRAD_ACCUM).backward()
        if step % GRAD_ACCUM == 0 or step == len(order):
            torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.0)
            optimizer.step();optimizer.zero_grad(set_to_none=True)
    scores,_=evaluate(dev_data)
    row={'epoch':ep,'train_loss':sum(losses)/len(losses),**scores}
    history.append(row)
    (OUT/'history.json').write_text(json.dumps(history,indent=2))
    print('Epoch',ep,'Dev Macro-F1:',scores['macro_f1'])
    if scores['macro_f1'] > best_score:
        best_score=scores['macro_f1'];stale=0
        torch.save({'model':model.state_dict(),'arch':ARCH,'method':METHOD,
                    'chunk_length':CHUNK_LENGTH,'max_chunks':MAX_CHUNKS,
                    'source':P1_REPO,'source_checkpoint':CHECKPOINT_FILE,
                    'dev_macro_f1':best_score},best_path)
    else:
        stale+=1
        if stale>=PATIENCE:break


Epoch 1:   0%|          | 0/37387 [00:00<?, ?it/s]

Predict:   0%|          | 0/1988 [00:00<?, ?it/s]

Epoch 1 Dev Macro-F1: 0.54729044971885


Epoch 2:   0%|          | 0/37387 [00:00<?, ?it/s]

Predict:   0%|          | 0/1988 [00:00<?, ?it/s]

Epoch 2 Dev Macro-F1: 0.5561859575696348


Epoch 3:   0%|          | 0/37387 [00:00<?, ?it/s]

Predict:   0%|          | 0/1988 [00:00<?, ?it/s]

Epoch 3 Dev Macro-F1: 0.5887505706645797


## 15 · Restore best checkpoint and evaluate development set

Always evaluate the **best** validation-selected model, not necessarily the last epoch. No hidden test labels are used.

In [15]:
# Reload the best saved Phase-4 weights, then compute final dev metrics.
model.load_state_dict(torch.load(best_path,map_location='cpu',weights_only=True)['model'])
final_dev,_=evaluate(dev_data)


Predict:   0%|          | 0/1988 [00:00<?, ?it/s]

## 16 · Predict every official test case

Test IDs retain their official order. Debug caps are rejected before packaging. Outputs are **integers 0/1**, exactly one per case.

In [16]:
# Run inference on the full official test set and validate output IDs/types.
_,test_predictions=evaluate(test_data)
if any(n is not None for n in (MAX_TRAIN,MAX_DEV,MAX_TEST)):
    raise RuntimeError('Debug caps enabled; disable them and run full data before saving test submission')
records=[{'id':case_id,'pred':int(pred)} for case_id,pred in zip(TEST_IDS,test_predictions)]
assert len(records)==len(test_cases)==len(TEST_IDS)
assert [r['id'] for r in records]==TEST_IDS
assert all(type(r['pred'])==int and r['pred'] in (0,1) for r in records)


Predict:   0%|          | 0/1517 [00:00<?, ?it/s]

## 17 · Write prediction.json and submission.zip

Codabench requires `prediction.json` at the **ZIP ROOT**, not inside a subdirectory. The cell checks the archive immediately.

In [17]:
# Save JSON list and ZIP with one top-level member.
(OUT/'prediction.json').write_text(json.dumps(records,indent=2),encoding='utf-8')
with zipfile.ZipFile(OUT/'submission.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write(OUT/'prediction.json',arcname='prediction.json')
with zipfile.ZipFile(OUT/'submission.zip') as z:
    assert z.namelist()==['prediction.json']
    assert json.loads(z.read('prediction.json'))==records


## 18 · Record metrics, training metadata, source checkpoint, and tokenizer

This information makes every experiment auditable. It does **not** treat development results as official competition scores.

In [18]:
# Write reproducibility evidence alongside checkpoint and submission files.
summary={'arch':ARCH,'method':METHOD,'source':P1_REPO,'source_checkpoint':CHECKPOINT_FILE,
         'train_cases':len(train_cases),'dev_cases':len(dev_cases),'test_cases':len(test_cases),
         'best_dev':final_dev,'parameters':sum(p.numel() for p in model.parameters()),
         'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad)}
(OUT/'metrics.json').write_text(json.dumps(summary,indent=2))
(OUT/'run_config.json').write_text(json.dumps({'chunk_length':CHUNK_LENGTH,'max_chunks':MAX_CHUNKS,
               'epochs':EPOCHS,'method':METHOD,'teacher_repo':TEACHER_REPO_NAME if METHOD=='kd' else None,
               'seed':SEED,'architecture_config':saved_cfg,'use_class_weights':USE_CLASS_WEIGHTS},indent=2))
if config_name: shutil.copy2(fetch(config_name),OUT/'phase1_architecture_config.json')
(OUT/'README.md').write_text(f'CS781 Phase 4 {ARCH} / {METHOD}\n\nOriginal model: {P1_REPO}/{CHECKPOINT_FILE}\n'
                             f'Official Phase 4 CJPE dataset only. Dev Macro-F1: {best_score}\n'
                             'The model is a case-level binary classifier (0/1). Full checkpoint and predictions are included.\n')
print('VALID submission:',OUT/'submission.zip','Dev metrics:',final_dev)


VALID submission: /kaggle/working/phase4_a1_full/submission.zip Dev metrics: {'macro_f1': 0.5887505706645797, 'precision_macro': 0.594111159664565, 'recall_macro': 0.5915492957746479, 'confusion_matrix': [[670, 324], [488, 506]]}


## 19 · Publish this run to a separate private Hugging Face repo

This optional cell is the only place that **writes** to Hugging Face. It creates a **private** repository under the authenticated account (detected automatically) and checks that every local file appeared remotely. It does not upload the official dataset.

In [19]:
# Publish only after training, metrics, and submission checks have passed.
if UPLOAD_TO_HF:
    api.create_repo(repo_id=OUT_REPO,repo_type='model',private=True,exist_ok=True)
    if not api.repo_info(OUT_REPO,repo_type='model').private:
        raise RuntimeError('Refusing upload: output Hugging Face repo is not private')
    api.upload_folder(repo_id=OUT_REPO,repo_type='model',folder_path=str(OUT),
                      commit_message=f'Phase 4 {ARCH} {METHOD} full experiment and predictions')
    expected={str(f.relative_to(OUT)).replace('\\','/') for f in OUT.rglob('*') if f.is_file()}
    remote=set(api.list_repo_files(OUT_REPO,repo_type='model'))
    missing=expected-remote
    if missing: raise RuntimeError(f'HF upload missing files: {sorted(missing)}')
    print('Hugging Face repo upload VERIFIED: https://huggingface.co/'+OUT_REPO)
else: print('HF upload disabled. Files remain in:',OUT)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Hugging Face repo upload VERIFIED: https://huggingface.co/P-261110044/cs781-phase4-a1-full


## 20 · Final output verification

All files must exist in the method-specific `/kaggle/working/phase4_<architecture>_<method>/` folder. To keep Kaggle outputs permanently, click **Save Version → Save & Run All** after you confirm the notebook works; this does not upload to Hugging Face by itself. The private HF upload is performed only in Step 19. This notebook does not need to be rerun for another method.

In [20]:
# Print the outputs after every step has succeeded.
required = ['best_model.pt', 'metrics.json', 'prediction.json', 'submission.zip', 'run_config.json']
for filename in required:
    file = OUT / filename
    print(('OK     ' if file.is_file() else 'MISSING'), file)
assert all((OUT / name).is_file() for name in required)
print('Output directory:', OUT)
print('HF repo (if uploaded): https://huggingface.co/' + OUT_REPO)
print('Experiment complete: this file trains ONLY the ' + METHOD + ' method.')

OK      /kaggle/working/phase4_a1_full/best_model.pt
OK      /kaggle/working/phase4_a1_full/metrics.json
OK      /kaggle/working/phase4_a1_full/prediction.json
OK      /kaggle/working/phase4_a1_full/submission.zip
OK      /kaggle/working/phase4_a1_full/run_config.json
Output directory: /kaggle/working/phase4_a1_full
HF repo (if uploaded): https://huggingface.co/P-261110044/cs781-phase4-a1-full
Experiment complete: this file trains ONLY the full method.
